In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import torch

model_id = 'mistralai/Mistral-7B-v0.3'
quant_config = BitsAndBytesConfig(
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quant_config
)
print(model)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
output = pipe("The capital of France is", max_new_tokens=50, top_k=5)
print("-------------")
for generation in output:
    print(generation['generated_text'])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from datasets import load_dataset
from transformers import BitsAndBytesConfig

# 1. Load tokenizer
model_id = "mistralai/Mistral-7B-v0.3"
output_dir = "models/qlora-mistral-10gb"
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# 2. Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="bfloat16"  # or torch.float16 if bfloat16 unsupported
)

# 3. Load quantized model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 4. Enable LoRA
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# 5. Dataset
dataset = load_dataset("Abirate/english_quotes", split="train[:5000]")  # very small subset

def tokenize(example):
    return tokenizer(example["quote"], truncation=True, padding="max_length", max_length=128)

tokenized = dataset.map(tokenize, batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 6. Training arguments
training_args = TrainingArguments(
    per_device_train_batch_size=16,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=10,
    output_dir=output_dir,
    save_strategy="epoch",
    fp16=True,  # use bf16=True if available
    report_to="none"
)

# 7. Train
trainer = Trainer(
    model=model,
    args=training_args,
    
    train_dataset=tokenized,
    data_collator=data_collator
)
trainer.train()


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/home/nitin/miniconda3/envs/dl/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.348100
20,1.353000
30,1.155200


/home/nitin/miniconda3/envs/dl/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=38, training_loss=1.261790250477038, metrics={'train_runtime': 869.9036, 'train_samples_per_second': 5.766, 'train_steps_per_second': 0.044, 'total_flos': 2.6936547144105984e+16, 'train_loss': 1.261790250477038, 'epoch': 1.9681528662420382})

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.3", device_map="auto", load_in_4bit=True)
model = PeftModel.from_pretrained(base_model, "./models/qlora-mistral-10gb/checkpoint-38")
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.3")
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DiffLlamaForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GlmForCausalLM', 'GotOcr2ForConditionalGeneration', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'GraniteForCausa

In [5]:
output = pipe("The capital of France is", max_new_tokens=50, top_k=5)
print("-------------")
for generation in output:
    print(generation['generated_text'])

/home/nitin/miniconda3/envs/dl/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


-------------
The capital of France is one of the most beautiful cities in the world. It is also one of the most expensive.

Paris is a city of contrasts. It is a city of art and culture, but also a city of poverty and crime. It is


In [7]:
import torch
inputs = "The capital of France is"
tokenized_inputs = tokenizer(inputs, return_tensors="pt").to(model.device)
print(tokenizer.convert_ids_to_tokens(tokenized_inputs['input_ids'][0]))
print(tokenized_inputs)

model.eval()
with torch.no_grad():
    outputs = model.generate(
        **tokenized_inputs, 
        max_new_tokens=100, 
        temperature=0.7, 
        top_k=50, 
        top_p=0.95, 
        do_sample=True, 
        repetition_penalty=1.2
    )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(generated_text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


['<s>', '▁The', '▁capital', '▁of', '▁France', '▁is']
{'input_ids': tensor([[   1, 1183, 6333, 1070, 5611, 1117]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]], device='cuda:0')}
The capital of France is a beautiful and historic city, but it can also be very expensive. If you’re planning on visiting Paris in the future, here are some tips to help make your trip more affordable:

## 1) Stay at an Airbnb instead of a hotel

Airbnbs are usually much cheaper than hotels – especially if you stay with friends or family who live nearby! You could even try couchsurfing for free accommodation! In addition to being less costly per night than


In [ ]:
from peft import PeftModel
model = PeftModel.from_pretrained(base_model, "./models/qlora-mistral-10gb/checkpoint-38")
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./models/Mistral-7B-v0.3-ft-Abirate-english_quotes")
tokenizer.save_pretrained("./models/Mistral-7B-v0.3-ft-Abirate-english_quotes")

('./models/Mistral-7B-v0.3-ft-Abirate-english_quotes/tokenizer_config.json',
 './models/Mistral-7B-v0.3-ft-Abirate-english_quotes/special_tokens_map.json',
 './models/Mistral-7B-v0.3-ft-Abirate-english_quotes/tokenizer.model',
 './models/Mistral-7B-v0.3-ft-Abirate-english_quotes/added_tokens.json',
 './models/Mistral-7B-v0.3-ft-Abirate-english_quotes/tokenizer.json')